# Advanced Python Problems with Solutions: Data Descriptors, Weak References, and Lifetime-Safe Storage

**20 substantial problems + 2 bonus labs | complete reference solutions | executable verification**

This workbook extends the attached lesson on storing descriptor-managed attributes in `weakref.WeakKeyDictionary`, dealing with unhashable objects, using `id()` plus weak-reference callbacks, and enabling weak references for slotted instances. Exercises add equality/hash pitfalls, descriptor precedence, validation, deletion, GC cycles, algorithmic costs, and a practical capstone.

**Runtime:** Python 3.9+; only the standard library. Run cells **top to bottom** in a fresh kernel. An assertion failure indicates a regression. All expected exceptions are explicitly caught, so successful execution should finish without errors. No dependencies, networking, or manual input.

**Safety/correctness note on the source:** Inspecting an address with `ctypes.c_long.from_address(id(obj))` *after the object has been freed* accesses invalid memory and is not a valid GC test. We do **not** use that technique. Instead, use `weakref.ref`, `gc.collect()`, and checks of descriptor storage. `del name` merely removes that name; collection occurs only when no strong references remain, with timing dependent on implementation and cycles. Identity addresses can be reused, so `id()` alone is not a permanent key.

**Learning sequence:** establish failures → implement an O(1) identity-aware weak store → build reusable descriptors → investigate lifetime hazards → finish with a validated unhashable slotted domain model.

## Notebook conventions and test helper

`expect_raises` makes the negative cases executable, including optional checks on the error message. The code deliberately never dereferences freed memory or assumes a particular `id()` value. CPython frequently destroys acyclic objects immediately, but `gc.collect()` makes cycle tests portable across normal garbage-collected Python implementations.

In [1]:
import gc
import weakref
from contextlib import contextmanager
from dataclasses import dataclass


@contextmanager
def expect_raises(exception_type, contains=None):
    """Assert that a block raises the expected exception (and optional message)."""
    try:
        yield
    except exception_type as exc:
        if contains is not None:
            assert contains in str(exc), (contains, str(exc))
    else:
        raise AssertionError(f"Expected {exception_type.__name__} to be raised")


def collect():
    """Collect unreachable cycles; never rely on a specific object address."""
    gc.collect()


print('Setup complete: only standard-library modules imported.')

Setup complete: only standard-library modules imported.


## Problem 01 — Trace the descriptor protocol and conversion

**Challenge.** Implement `IntegerValue` with an ordinary dictionary, including class-level access. Prove that two owners keep different values and determine how `int()` handles a float and a Boolean. Explain why this is only a baseline, not a production implementation.

**Solution reasoning.** `__get__(None, owner)` should return the descriptor. Python calls a data descriptor’s `__set__` when assigning an instance attribute. The lesson uses `int(value)`, which **truncates** floats and accepts `bool`; that conversion is a design choice, not strict integer validation. A standard dictionary has **strong keys**.

In [2]:
# Problem 01: reference solution
class BaselineIntegerValue:
    def __init__(self):
        self.values = {}

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return self.values.get(instance)

    def __set__(self, instance, value):
        self.values[instance] = int(value)


class BaselinePoint:
    x = BaselineIntegerValue()
    y = BaselineIntegerValue()

In [3]:
# Problem 01: executable checks
p, q = BaselinePoint(), BaselinePoint()
p.x, p.y, q.x = 19.9, True, -2.9
assert (p.x, p.y, q.x, q.y) == (19, 1, -2, None)
assert BaselinePoint.x is BaselinePoint.__dict__['x']
assert BaselinePoint.x is not BaselinePoint.y
BaselinePoint.x.values.clear()
BaselinePoint.y.values.clear()
del p, q
print("Problem 01: PASS")

Problem 01: PASS


## Problem 02 — Reproduce and fix a strong-key lifetime leak

**Challenge.** Create a descriptor-backed owner, hold only its weak reference, remove the local owner name, and show that the descriptor’s normal dictionary keeps the owner alive. Demonstrate a safe manual cleanup.

**Solution reasoning.** A dictionary holds references to keys. Thus a class-owned descriptor can inadvertently prolong every instance’s lifetime. The weak reference observes collection without retaining the referent; `clear()` releases the remaining strong key.

In [4]:
# Problem 02: reference solution
class LeakyInteger:
    def __init__(self):
        self.values = {}

    def __get__(self, instance, owner):
        return self if instance is None else self.values[instance]

    def __set__(self, instance, value):
        self.values[instance] = int(value)


class LeakyPoint:
    x = LeakyInteger()

In [5]:
# Problem 02: executable checks
owner = LeakyPoint()
owner.x = 47
owner_ref = weakref.ref(owner)
del owner
collect()
assert owner_ref() is not None          # descriptor is still a strong owner
assert len(LeakyPoint.x.values) == 1
LeakyPoint.x.values.clear()
collect()
assert owner_ref() is None              # manually release the leaked owner
print("Problem 02: PASS")

Problem 02: PASS


## Problem 03 — Replace strong keys with WeakKeyDictionary

**Challenge.** Refactor the baseline using `weakref.WeakKeyDictionary`. Show independent instance values, class access, and automatic entry removal when the instance becomes unreachable.

**Solution reasoning.** `WeakKeyDictionary` uses weak keys, so it does not keep a weak-referenceable instance alive **provided its stored value does not point back to that instance**. It still requires hashable keys, a limitation explored next.

In [6]:
# Problem 03: reference solution
class WeakInteger:
    def __init__(self):
        self.values = weakref.WeakKeyDictionary()

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return self.values[instance]
        except KeyError:
            raise AttributeError('integer value is not set') from None

    def __set__(self, instance, value):
        self.values[instance] = int(value)


class WeakPoint:
    x = WeakInteger()

In [7]:
# Problem 03: executable checks
a, b = WeakPoint(), WeakPoint()
a.x, b.x = 21.7, 82.9
assert (a.x, b.x) == (21, 82)
assert WeakPoint.x is WeakPoint.__dict__['x']
assert len(WeakPoint.x.values) == 2
ref_a, ref_b = weakref.ref(a), weakref.ref(b)
del a
collect()
assert ref_a() is None and ref_b() is not None
assert len(WeakPoint.x.values) == 1
del b
collect()
assert len(WeakPoint.x.values) == 0
print("Problem 03: PASS")

Problem 03: PASS


## Problem 04 — Separate weak-referenceability from hashability

**Challenge.** Make an owner unhashable by implementing equality without `__hash__`. Confirm that a weak reference can be made even though it cannot serve as a `WeakKeyDictionary` key.

**Solution reasoning.** Weak-referenceability and hashability are separate properties. An instance with normal `__dict__` is typically weak-referenceable, but defining `__eq__` usually sets `__hash__ = None`. A `weakref.ref(obj)` may be created, yet hashing the weak reference raises `TypeError` when the referent is unhashable.

In [8]:
# Problem 04: reference solution
class UnhashablePoint:
    x = WeakInteger()

    def __init__(self, coordinate):
        self.coordinate = coordinate

    def __eq__(self, other):
        return isinstance(other, UnhashablePoint) and self.coordinate == other.coordinate

In [9]:
# Problem 04: executable checks
point = UnhashablePoint(8)
assert point == UnhashablePoint(8)
assert UnhashablePoint.__hash__ is None
reference = weakref.ref(point)
assert reference() is point
with expect_raises(TypeError):
    hash(point)
with expect_raises(TypeError):
    hash(reference)
with expect_raises(TypeError):
    point.x = 20
assert len(UnhashablePoint.x.values) == 0
print("Problem 04: PASS")

Problem 04: PASS


## Problem 05 — Discover the equal-but-distinct weak-key trap

**Challenge.** Insert two different but equal hashable owners into `WeakKeyDictionary`, using different values. Observe the entry count and behavior after the *original* key dies. Why can this violate an identity-based descriptor’s semantics?

**Solution reasoning.** `WeakKeyDictionary` respects referent equality and hash, not owner identity. Inserting an equal second key replaces the value but can retain the original weak-key object. When that original owner dies, the entry may disappear while the second owner remains alive. Identity-based per-instance storage must avoid this behavior.

In [10]:
# Problem 05: reference solution
@dataclass(frozen=True)
class EqualKey:
    number: int


equal_key_values = weakref.WeakKeyDictionary()

In [11]:
# Problem 05: executable checks
first, second = EqualKey(5), EqualKey(5)
assert first == second and first is not second
equal_key_values[first] = 'first owner'
equal_key_values[second] = 'second owner'
assert len(equal_key_values) == 1
assert equal_key_values[first] == equal_key_values[second] == 'second owner'
first_ref = weakref.ref(first)
del first
collect()
assert first_ref() is None
assert second is not None and len(equal_key_values) == 0
with expect_raises(KeyError):
    _ = equal_key_values[second]
del second
print("Problem 05: PASS")

Problem 05: PASS


## Problem 06 — Expose stale state in an id-only dictionary

**Challenge.** Index descriptor state by `id(instance)` so unhashable objects work. Delete an instance, then demonstrate that its integer-key entry is not removed. Do not assume you can force ID reuse.

**Solution reasoning.** An integer key does not retain the instance, but it also receives no lifetime notification. The abandoned entry wastes memory and could be mistaken for another object if an address is reused. Reading memory at the old address through `ctypes` would be unsafe and proves nothing.

In [12]:
# Problem 06: reference solution
class IdOnlyInteger:
    def __init__(self):
        self.values = {}

    def __get__(self, instance, owner):
        return self if instance is None else self.values.get(id(instance))

    def __set__(self, instance, value):
        self.values[id(instance)] = int(value)


class IdOnlyPoint:
    x = IdOnlyInteger()
    __hash__ = None

In [13]:
# Problem 06: executable checks
old = IdOnlyPoint()
old.x = 99.9
old_id = id(old)
old_ref = weakref.ref(old)
assert old.x == 99
del old
collect()
assert old_ref() is None
assert IdOnlyPoint.x.values[old_id] == 99  # stale state persists
IdOnlyPoint.x.values.clear()             # only a manual fix
print("Problem 06: PASS")

Problem 06: PASS


---
## Reusable core: identity-indexed weak storage

The next reference implementation is the engine for several remaining solutions. Its invariants are: (1) `id()` is only a lookup accelerator while an entry retains a weak reference; (2) `weak_ref() is instance` confirms identity on lookup and deletion; (3) a callback captures **only the integer ID and a weak reference to the store**, never the owner; (4) a callback removes the entry only if it still holds its *exact* weak-reference object; (5) deletion is explicit; (6) no linear scan is needed. A stored **value** must not keep its owner alive. This is a single-threaded teaching implementation, not a synchronization primitive.

## Problem 07 — Implement an O(1) identity-aware weak store

**Challenge.** Write a storage class supporting `set(owner, value)`, `get(owner)`, `delete(owner)`, and `len(store)` for unhashable, weak-referenceable owners. Register an O(1) weakref callback without capturing owners and use an identity guard.

**Solution reasoning.** Store `{id(owner): (weakref.ref(owner, callback), value)}`. Make the callback hold `weakref.ref(store)` instead of a strong reference to the store and look up a single integer key. Check `record[0] is dead_ref` before removal to prevent an obsolete callback from deleting replacement state. Reject unsupported owners *before* updating storage.

In [14]:
# Problem 07: reference solution
class IdentityWeakStore:
    """Single-threaded, weak-referenceable, identity-based per-owner storage."""

    def __init__(self):
        self._data = {}  # int -> (weakref.ref(owner), payload)

    def _remove_if_current(self, key, dead_ref):
        record = self._data.get(key)
        if record is not None and record[0] is dead_ref:
            del self._data[key]

    def set(self, owner, value):
        key = id(owner)
        storage_ref = weakref.ref(self)

        def on_collect(dead_ref, key=key, storage_ref=storage_ref):
            storage = storage_ref()
            if storage is not None:
                storage._remove_if_current(key, dead_ref)

        # Construct the weak ref first: failure must not change old state.
        owner_ref = weakref.ref(owner, on_collect)
        self._data[key] = (owner_ref, value)

    def get(self, owner):
        record = self._data.get(id(owner))
        if record is None or record[0]() is not owner:
            raise KeyError('no value for this exact owner')
        return record[1]

    def delete(self, owner):
        key = id(owner)
        record = self._data.get(key)
        if record is None or record[0]() is not owner:
            raise KeyError('no value for this exact owner')
        del self._data[key]

    def __len__(self):
        return len(self._data)


class IdentityOwner:
    __slots__ = ('__weakref__',)
    __hash__ = None

In [15]:
# Problem 07: executable checks
store = IdentityWeakStore()
one, two = IdentityOwner(), IdentityOwner()
store.set(one, 10)
store.set(two, None)  # None is a legitimate stored value, not an unset marker
assert store.get(one) == 10 and store.get(two) is None
assert len(store) == 2
with expect_raises(KeyError):
    store.get(IdentityOwner())
one_ref, two_ref = weakref.ref(one), weakref.ref(two)
del one
collect()
assert one_ref() is None and two_ref() is two
assert len(store) == 1
store.delete(two)
assert len(store) == 0
with expect_raises(KeyError):
    store.delete(two)
del two, store
print("Problem 07: PASS")

Problem 07: PASS


## Problem 08 — Prevent a stale callback from deleting fresh state

**Challenge.** Assign to the *same* owner twice and retain the old weakref object. Simulate a delayed callback for the old ref; prove the newer value survives. Also check owner isolation and full cleanup.

**Solution reasoning.** Repeated writes replace the ref/record associated with an ID. Even though real callbacks normally run when the owner dies, the identity guard is a defensive invariant against stale or reordered cleanup events. Call the store’s cleanup method explicitly to test it deterministically.

In [16]:
# Problem 08: reference solution
replacement_store = IdentityWeakStore()
owner_a, owner_b = IdentityOwner(), IdentityOwner()
replacement_store.set(owner_a, 'old')
key_a = id(owner_a)
old_weak_ref = replacement_store._data[key_a][0]
replacement_store.set(owner_a, 'new')
new_weak_ref = replacement_store._data[key_a][0]
replacement_store.set(owner_b, 'other')

In [17]:
# Problem 08: executable checks
assert old_weak_ref is not new_weak_ref
replacement_store._remove_if_current(key_a, old_weak_ref)
assert replacement_store.get(owner_a) == 'new'
assert replacement_store.get(owner_b) == 'other'
assert len(replacement_store) == 2
ra, rb = weakref.ref(owner_a), weakref.ref(owner_b)
del owner_a, owner_b
collect()
assert ra() is None and rb() is None
assert len(replacement_store) == 0
# old_weak_ref/new_weak_ref do not keep the owners alive.
del old_weak_ref, new_weak_ref, replacement_store
print("Problem 08: PASS")

Problem 08: PASS


## Problem 09 — Build a strict data descriptor with deletion

**Challenge.** Adapt the identity store into a reusable integer descriptor. Require actual integers (reject `bool` and floats), implement `__set_name__`, return the descriptor on class access, raise `AttributeError` for unset values, and support `del obj.field`.

**Solution reasoning.** A descriptor’s validation and storage semantics are distinct. `type(value) is int` intentionally excludes `bool`, which `isinstance(True, int)` would accept. `__delete__` should translate an absent record into `AttributeError`, consistent with normal missing attributes. The descriptor instance owns one separate store.

In [18]:
# Problem 09: reference solution
class StrictInteger:
    def __init__(self):
        self.store = IdentityWeakStore()
        self.name = '<unbound>'

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return self.store.get(instance)
        except KeyError:
            raise AttributeError(f'{self.name} has not been assigned') from None

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError(f'{self.name} must be an int (not bool)')
        self.store.set(instance, value)

    def __delete__(self, instance):
        try:
            self.store.delete(instance)
        except KeyError:
            raise AttributeError(f'{self.name} has not been assigned') from None


class StrictPoint:
    __slots__ = ('__weakref__',)
    __hash__ = None
    x = StrictInteger()

In [19]:
# Problem 09: executable checks
point = StrictPoint()
assert StrictPoint.x.name == 'x'
assert StrictPoint.x is StrictPoint.__dict__['x']
with expect_raises(AttributeError, 'x has not been assigned'):
    _ = point.x
point.x = -12
assert point.x == -12
for invalid in (True, 4.5, '12', None):
    with expect_raises(TypeError):
        point.x = invalid
    assert point.x == -12  # failed validation leaves the old value intact
del point.x
with expect_raises(AttributeError):
    _ = point.x
with expect_raises(AttributeError):
    del point.x
assert len(StrictPoint.x.store) == 0
print("Problem 09: PASS")

Problem 09: PASS


## Problem 10 — Create independent length-validated string fields

**Challenge.** Create a `ValidString(min_length, max_length)` descriptor using the identity store. Validate its constructor, the value’s type, and inclusive length bounds. Use two descriptors on a slotted, unhashable `Person` and confirm isolation across people and fields.

**Solution reasoning.** Use one store per descriptor object, so different fields cannot overwrite one another. Use `__set_name__` for useful errors. Preserve an existing value when validation fails: validate *before* storage. `__slots__ = ('__weakref__',)` is sufficient because the descriptor stores field values externally.

In [20]:
# Problem 10: reference solution
class ValidString:
    def __init__(self, min_length=0, max_length=255):
        if (type(min_length) is not int or type(max_length) is not int
                or min_length < 0 or max_length < min_length):
            raise ValueError('length bounds must satisfy 0 <= min_length <= max_length')
        self.min_length = min_length
        self.max_length = max_length
        self.name = '<unbound>'
        self.store = IdentityWeakStore()

    def __set_name__(self, owner, name):
        self.name = name

    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f'{self.name} must be a string')
        if not self.min_length <= len(value) <= self.max_length:
            raise ValueError(
                f'{self.name} length must be from {self.min_length} to {self.max_length}'
            )
        return value

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return self.store.get(instance)
        except KeyError:
            raise AttributeError(f'{self.name} has not been assigned') from None

    def __set__(self, instance, value):
        self.store.set(instance, self.validate(value))

    def __delete__(self, instance):
        try:
            self.store.delete(instance)
        except KeyError:
            raise AttributeError(f'{self.name} has not been assigned') from None


class Person:
    __slots__ = ('__weakref__',)
    __hash__ = None
    first_name = ValidString(1, 15)
    last_name = ValidString(1, 20)

In [21]:
# Problem 10: executable checks
with expect_raises(ValueError):
    ValidString(-1, 4)
with expect_raises(ValueError):
    ValidString(5, 3)
p1, p2 = Person(), Person()
p1.first_name, p1.last_name = 'Ada', 'Lovelace'
p2.first_name, p2.last_name = 'Grace', 'Hopper'
assert (p1.first_name, p1.last_name) == ('Ada', 'Lovelace')
assert (p2.first_name, p2.last_name) == ('Grace', 'Hopper')
assert Person.first_name.store is not Person.last_name.store
assert len(Person.first_name.store) == len(Person.last_name.store) == 2
with expect_raises(TypeError):
    p1.first_name = 100
with expect_raises(ValueError):
    p1.last_name = ''
assert p1.first_name == 'Ada' and p1.last_name == 'Lovelace'
print("Problem 10: PASS")

Problem 10: PASS


## Problem 11 — Avoid partially updated multi-field operations

**Challenge.** Implement a safe two-field rename for the `Person` above: if either input is invalid, **neither** field changes. Demonstrate the difference between validating both first and blindly assigning sequentially.

**Solution reasoning.** Each descriptor validates a single assignment atomically, but two sequential assignments are not automatically one transaction. Prevalidate *both* inputs before committing either; for this restricted in-memory example, after validation the supported owners can accept both assignments. Larger systems may require rollback or transactional storage.

In [22]:
# Problem 11: reference solution
def safe_rename(person, first, last):
    validated_first = type(person).first_name.validate(first)
    validated_last = type(person).last_name.validate(last)
    person.first_name = validated_first
    person.last_name = validated_last

In [23]:
# Problem 11: executable checks
original = (p1.first_name, p1.last_name)
with expect_raises(ValueError, 'last_name'):
    safe_rename(p1, 'Augusta', '')
assert (p1.first_name, p1.last_name) == original
with expect_raises(TypeError, 'first_name'):
    safe_rename(p1, None, 'Byron')
assert (p1.first_name, p1.last_name) == original
safe_rename(p1, 'Augusta', 'Byron')
assert (p1.first_name, p1.last_name) == ('Augusta', 'Byron')
print("Problem 11: PASS")

Problem 11: PASS


## Problem 12 — Verify data-descriptor precedence over __dict__

**Challenge.** Use a non-slotted class with a data descriptor and manually insert a shadowing key with the same name into its instance dictionary. Which value wins? Does deleting the descriptor-managed value reveal the shadow?

**Solution reasoning.** A descriptor with `__set__` (or `__delete__`) is a **data descriptor**: it takes precedence over the instance dictionary on ordinary attribute access. Even when its store has no entry, `__get__` is still called and can raise `AttributeError`; the manually written dictionary entry does not transparently become active.

In [24]:
# Problem 12: reference solution
class DictPerson:
    name = ValidString(1, 30)


dict_person = DictPerson()
dict_person.name = 'descriptor value'
dict_person.__dict__['name'] = 'manual shadow'

In [25]:
# Problem 12: executable checks
assert dict_person.name == 'descriptor value'
assert vars(dict_person)['name'] == 'manual shadow'
assert DictPerson.name is DictPerson.__dict__['name']
del dict_person.name
with expect_raises(AttributeError):
    _ = dict_person.name
assert dict_person.__dict__['name'] == 'manual shadow'
del dict_person
collect()
assert len(DictPerson.name.store) == 0
print("Problem 12: PASS")

Problem 12: PASS


## Problem 13 — Enable weak references in slot-only owners

**Challenge.** Demonstrate why a class declaring only `__slots__ = ('label',)` cannot be used with the identity weak store. Fix it by adding `__weakref__`. Verify the corrected instance still has no `__dict__` and that subclasses inherit weak-reference support.

**Solution reasoning.** A class defining `__slots__` without a `__dict__` or inherited weak-reference support generally must include `__weakref__` explicitly. `__weakref__` is an implementation-facing attribute, not a place to put descriptor values. Do not declare this slot again in a subclass that already inherits the capability.

In [26]:
# Problem 13: reference solution
class SlotWithoutWeak:
    __slots__ = ('label',)


class SlotWithWeak:
    __slots__ = ('label', '__weakref__')
    count = StrictInteger()


class ChildWithWeak(SlotWithWeak):
    __slots__ = ()

In [27]:
# Problem 13: executable checks
unusable = SlotWithoutWeak()
with expect_raises(TypeError):
    weakref.ref(unusable)
with expect_raises(TypeError):
    IdentityWeakStore().set(unusable, 'not supported')
usable = SlotWithWeak()
usable.label = 'ready'
usable.count = 8
assert usable.count == 8 and usable.label == 'ready'
assert weakref.ref(usable)() is usable
assert not hasattr(usable, '__dict__')
child = ChildWithWeak()
child.count = 9
assert weakref.ref(child)() is child and child.count == 9
assert len(SlotWithWeak.count.store) == 2
del usable, child
collect()
assert len(SlotWithWeak.count.store) == 0
print("Problem 13: PASS")

Problem 13: PASS


## Problem 14 — Collect cyclic owners without relying on refcounts

**Challenge.** Construct an owner whose `peer` points to itself, give it descriptor-managed data, and remove the external reference. Use explicit garbage collection to verify both the owner and its dictionary entry disappear.

**Solution reasoning.** A self-cycle is not necessarily destroyed immediately when its local name is deleted. The weakref callback should still clean the store once cyclic GC determines the referent is unreachable. Avoid exact refcount and GC timing assumptions.

In [28]:
# Problem 14: reference solution
class CyclicOwner:
    __slots__ = ('peer', '__weakref__')
    __hash__ = None
    count = StrictInteger()


cyclic = CyclicOwner()
cyclic.peer = cyclic
cyclic.count = 123
cyclic_ref = weakref.ref(cyclic)
assert len(CyclicOwner.count.store) == 1

In [29]:
# Problem 14: executable checks
del cyclic
collect()
assert cyclic_ref() is None
assert len(CyclicOwner.count.store) == 0
print("Problem 14: PASS")

Problem 14: PASS


## Problem 15 — Detect values that retain their own weak keys

**Challenge.** Store a payload that holds a **strong reference back to its owner**. Show that an otherwise correct identity weak store cannot collect that owner. Break the back-reference by deleting the entry, then verify cleanup.

**Solution reasoning.** Weak keys alone cannot prevent retention if the associated *value* strongly references the owner. The path `store → value → owner` remains a strong reachability chain, so the owner’s weakref does not die and the callback cannot execute. This is an architectural restriction on stored values.

In [30]:
# Problem 15: reference solution
class OwnerPayload:
    def __init__(self, owner):
        self.owner = owner


backref_store = IdentityWeakStore()
backref_owner = IdentityOwner()
backref_owner_ref = weakref.ref(backref_owner)
backref_store.set(backref_owner, OwnerPayload(backref_owner))

In [31]:
# Problem 15: executable checks
del backref_owner
collect()
assert backref_owner_ref() is not None  # value -> owner keeps it alive
assert len(backref_store) == 1
backref_store.delete(backref_owner_ref())  # temporary strong ref does not escape
collect()
assert backref_owner_ref() is None
assert len(backref_store) == 0
print("Problem 15: PASS")

Problem 15: PASS


## Problem 16 — Find a hidden strong owner captured by a callback

**Challenge.** Build an intentionally broken weak-reference callback that closes over its owner. Demonstrate the leak, then state how the reference implementation from Problem 7 avoids it.

**Solution reasoning.** A weakref callback is kept alive by its weakref; if the callback closure itself retains the owner, the chain `store → weakref → callback → owner` prevents collection. Our safe callback captures only the integer key and a weak reference to the storage object, never the owner.

In [32]:
# Problem 16: reference solution
class BrokenCallbackStore:
    def __init__(self):
        self.data = {}

    def set(self, owner, value):
        def on_collect(_dead_ref):
            # Captures `owner` strongly: DON'T DO THIS in production.
            self.data.pop(id(owner), None)

        self.data[id(owner)] = (weakref.ref(owner, on_collect), value)


broken = BrokenCallbackStore()
broken_owner = IdentityOwner()
broken_ref = weakref.ref(broken_owner)
broken.set(broken_owner, 'value')

In [33]:
# Problem 16: executable checks
del broken_owner
collect()
assert broken_ref() is not None
assert len(broken.data) == 1
broken.data.clear()
collect()
assert broken_ref() is None
assert len(broken.data) == 0
print("Problem 16: PASS")

Problem 16: PASS


## Problem 17 — Reproduce the same back-reference bug with WeakKeyDictionary

**Challenge.** Place a key directly inside its own `WeakKeyDictionary` value. Remove the external key reference and explain why the entry does not disappear until the mapping releases the value.

**Solution reasoning.** This flaw affects standard weak-key dictionaries too: their values are held strongly. A weak key only helps if the value graph does not keep the key reachable. The weak-key API does not automatically break references *inside values*.

In [34]:
# Problem 17: reference solution
class HashableOwner:
    pass


weak_mapping = weakref.WeakKeyDictionary()
weak_owner = HashableOwner()
weak_owner_ref = weakref.ref(weak_owner)
weak_mapping[weak_owner] = {'owner': weak_owner}

In [35]:
# Problem 17: executable checks
del weak_owner
collect()
assert weak_owner_ref() is not None
assert len(weak_mapping) == 1
weak_mapping.clear()
collect()
assert weak_owner_ref() is None
assert len(weak_mapping) == 0
print('Problem 17: PASS')

Problem 17: PASS


## Problem 18 — Measure the reverse-lookup cost structurally

**Challenge.** Implement the lesson’s cleanup-by-scanning approach and count entries inspected after collecting the most recently inserted of several owners. Contrast its O(n) cleanup with Problem 7’s single-key lookup (expected O(1) average dictionary work). Do not rely on noisy microbenchmarks.

**Solution reasoning.** The source performs a reverse lookup through every entry to find the dead weakref. The following version counts exact comparisons, which demonstrates the scaling problem deterministically. Average O(1) for dictionary access assumes ordinary hash-table behavior; complete storage/GC complexity is not guaranteed constant.

In [36]:
# Problem 18: reference solution
class ScanningStore:
    def __init__(self):
        self.data = {}
        self.comparisons = 0

    def set(self, owner, value):
        self.data[id(owner)] = (weakref.ref(owner, self._remove), value)

    def _remove(self, dead_ref):
        for key, (reference, _value) in list(self.data.items()):
            self.comparisons += 1
            if reference is dead_ref:
                del self.data[key]
                return


scan = ScanningStore()
scan_owners = [IdentityOwner() for _ in range(250)]
for index, owner in enumerate(scan_owners):
    scan.set(owner, index)
del owner  # avoid retaining the last owner through the loop variable

In [37]:
# Problem 18: executable checks
assert len(scan.data) == 250
last_owner = scan_owners.pop()
last_ref = weakref.ref(last_owner)
del last_owner
collect()
assert last_ref() is None
assert scan.comparisons == 250   # last insertion: linear scan of all 250
assert len(scan.data) == 249
del scan_owners
collect()
assert len(scan.data) == 0
print("Problem 18: PASS")

Problem 18: PASS


## Problem 19 — Investigate mutable hashes in weak-key mappings

**Challenge.** Build an owner whose hash depends on mutable data, insert it into a `WeakKeyDictionary`, mutate the data, and attempt a lookup. Describe the invariants that hash-based mappings expect.

**Solution reasoning.** A dictionary assumes the hash (and equality relation) of a key remain consistent while stored. Mutating a hash-relevant field makes the normal lookup target another bucket, so retrieval can fail even when an old key is physically present. This is another reason to avoid `WeakKeyDictionary` for arbitrary mutable value-equality owners.

In [38]:
# Problem 19: reference solution
class MutableHashOwner:
    def __init__(self, number):
        self.number = number

    def __eq__(self, other):
        return isinstance(other, MutableHashOwner) and self.number == other.number

    def __hash__(self):
        return hash(self.number)


mutable_mapping = weakref.WeakKeyDictionary()
mutable = MutableHashOwner(101)
mutable_mapping[mutable] = 'record'

In [39]:
# Problem 19: executable checks
assert mutable_mapping[mutable] == 'record'
mutable.number = 202
assert mutable not in mutable_mapping
with expect_raises(KeyError):
    _ = mutable_mapping[mutable]
assert len(mutable_mapping) == 1  # retained entry is currently hard to retrieve
mutable_ref = weakref.ref(mutable)
del mutable
collect()
assert mutable_ref() is None
assert len(mutable_mapping) == 0
print("Problem 19: PASS")

Problem 19: PASS


## Problem 20 — Capstone: a validated, slot-only, unhashable inventory item

**Challenge.** Create an `InventoryItem` with a SKU string (3–12 characters) and quantity (strict integer, 0–500). Use separate external weak stores so there is no `__dict__`. Define value equality to make the owner unhashable, provide atomic `adjust(delta)` behavior, and verify validation, owner independence, and cleanup.

**Solution reasoning.** Combine `ValidString` and a bounded-integer descriptor built on `IdentityWeakStore`. Validate before storing, and calculate/check the new quantity before committing. Explicitly turn off hashing because equality is value-based; an ID-indexed weak store remains safe even for two equal but distinct owners. Validate all errors and free all owners at the end.

In [40]:
# Problem 20: reference solution
class BoundedInteger:
    def __init__(self, minimum, maximum):
        if (type(minimum) is not int or type(maximum) is not int
                or maximum < minimum):
            raise ValueError('invalid integer bounds')
        self.minimum, self.maximum = minimum, maximum
        self.name = '<unbound>'
        self.store = IdentityWeakStore()

    def __set_name__(self, owner, name):
        self.name = name

    def validate(self, value):
        if type(value) is not int:
            raise TypeError(f'{self.name} must be an integer, not bool')
        if not self.minimum <= value <= self.maximum:
            raise ValueError(f'{self.name} must be between {self.minimum} and {self.maximum}')
        return value

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return self.store.get(instance)
        except KeyError:
            raise AttributeError(f'{self.name} has not been assigned') from None

    def __set__(self, instance, value):
        self.store.set(instance, self.validate(value))

    def __delete__(self, instance):
        try:
            self.store.delete(instance)
        except KeyError:
            raise AttributeError(f'{self.name} has not been assigned') from None


class InventoryItem:
    __slots__ = ('__weakref__',)
    sku = ValidString(3, 12)
    quantity = BoundedInteger(0, 500)
    __hash__ = None

    def __init__(self, sku, quantity):
        # Validate all fields before committing either.
        checked_sku = type(self).sku.validate(sku)
        checked_quantity = type(self).quantity.validate(quantity)
        self.sku = checked_sku
        self.quantity = checked_quantity

    def __eq__(self, other):
        if not isinstance(other, InventoryItem):
            return NotImplemented
        return (self.sku, self.quantity) == (other.sku, other.quantity)

    def adjust(self, delta):
        if type(delta) is not int:
            raise TypeError('delta must be an integer, not bool')
        proposed = type(self).quantity.validate(self.quantity + delta)
        self.quantity = proposed
        return proposed

In [41]:
# Problem 20: executable checks
item_a = InventoryItem('ABC-123', 50)
item_b = InventoryItem('ABC-123', 50)
assert item_a is not item_b and item_a == item_b
assert InventoryItem.__hash__ is None
assert not hasattr(item_a, '__dict__')
with expect_raises(TypeError):
    hash(item_a)
assert item_a.adjust(+10) == 60
assert (item_a.quantity, item_b.quantity) == (60, 50)
for bad_delta, error in ((True, TypeError), (3.2, TypeError), (-61, ValueError), (441, ValueError)):
    with expect_raises(error):
        item_a.adjust(bad_delta)
    assert item_a.quantity == 60
for bad_sku, error in (('x', ValueError), ('x' * 13, ValueError), (99, TypeError)):
    with expect_raises(error):
        InventoryItem(bad_sku, 2)
for bad_quantity, error in ((True, TypeError), (-1, ValueError), (501, ValueError)):
    with expect_raises(error):
        InventoryItem('SAFE', bad_quantity)
assert len(InventoryItem.sku.store) == len(InventoryItem.quantity.store) == 2
ref_a, ref_b = weakref.ref(item_a), weakref.ref(item_b)
del item_a, item_b
collect()
assert ref_a() is None and ref_b() is None
assert len(InventoryItem.sku.store) == len(InventoryItem.quantity.store) == 0
print("Problem 20: PASS")

Problem 20: PASS


---
## Bonus lab A — Choose instance-dictionary storage when possible

The source’s external-storage approach is worthwhile for weak-referenceable slot-only owners, including unhashable ones. If your instances **have a `__dict__`**, storing data there is usually simpler: no weak references, no class-level owner registry, no cleanup callbacks, and no artificial weak-referenceability requirement. `__set_name__` supplies a backing key. This example intentionally requires `__dict__`; it does not claim universal applicability.

In [42]:
class DictBackedString:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f'_validated_{name}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return vars(instance)[self.storage_name]
        except KeyError:
            raise AttributeError(f'{self.name} has not been assigned') from None

    def __set__(self, instance, value):
        if not isinstance(value, str) or not value:
            raise ValueError(f'{self.name} must be a nonempty string')
        vars(instance)[self.storage_name] = value

    def __delete__(self, instance):
        try:
            del vars(instance)[self.storage_name]
        except KeyError:
            raise AttributeError(f'{self.name} has not been assigned') from None


class DictEnabledOwner:
    __slots__ = ('__dict__',)  # no __weakref__ needed for instance dictionary storage
    __hash__ = None
    name = DictBackedString()


dict_owner = DictEnabledOwner()
dict_owner.name = 'Stored on instance'
assert dict_owner.__dict__ == {'_validated_name': 'Stored on instance'}
assert dict_owner.name == 'Stored on instance'
del dict_owner.name
with expect_raises(AttributeError):
    _ = dict_owner.name
print('Bonus lab A: PASS')

Bonus lab A: PASS


## Bonus lab B — Stress-test separation and automatic cleanup

Create hundreds of owners with two descriptor fields. Keep only a small set alive, check exact store sizes, and then release the survivors. No raw addresses, sleep calls, unreliable performance timing, or `ctypes` are required.

In [43]:
class StressOwner:
    __slots__ = ('__weakref__',)
    __hash__ = None
    left = StrictInteger()
    right = StrictInteger()


def build_short_lived_owners(count):
    observed = []
    for number in range(count):
        owner = StressOwner()
        owner.left = number
        owner.right = -number
        observed.append(weakref.ref(owner))
    # The final `owner` name also disappears when this function returns.
    return observed


survivors = [StressOwner() for _ in range(20)]
for index, owner in enumerate(survivors):
    owner.left = index
    owner.right = index * 2
del owner
references = build_short_lived_owners(400)
collect()
assert all(reference() is None for reference in references)
assert len(StressOwner.left.store) == len(StressOwner.right.store) == 20
assert [(owner.left, owner.right) for owner in survivors] == [
    (index, index * 2) for index in range(20)
]
del survivors
collect()
assert len(StressOwner.left.store) == len(StressOwner.right.store) == 0
print('Bonus lab B: PASS — isolated values and automatic cleanup.')

Bonus lab B: PASS — isolated values and automatic cleanup.


---
## Reference decision guide and further challenges

| Owner / requirement | Useful approach | Main caution |
|---|---|---|
| Instance has a writable `__dict__` | `__set_name__` + instance-backed storage | Choose a nonconflicting backing name; account for inherited descriptors. |
| Hashable, weak-referenceable owner with identity-based equality | `WeakKeyDictionary` | Values must not refer strongly to their keys. |
| Unhashable but weak-referenceable slot-only owner | Guarded ID → weakref/value store | More code; each value and callback must not retain its owner. |
| Value-equality owners that compare equal | Guarded identity store or instance dictionary | A `WeakKeyDictionary` may merge owners. |
| Owner cannot be weakly referenced and has no `__dict__` | Change the class design or provide an explicit lifecycle API | No general automatic external lifetime cleanup is available under those constraints. |

**Remaining engineering constraints:** The reference `IdentityWeakStore` is intentionally single-threaded. If owners are accessed across threads, protect state mutations and callback cleanup with appropriate locking and carefully document reentrancy and lock ordering. Owners with custom attribute lookup, slots inherited through complex hierarchies, or destructors require additional integration testing. Keys based on `id()` must *always* be paired with live identity verification and deletion guards, not used as permanent identifiers.

**Independent extension exercises:** (1) Add a configurable default factory without materializing defaults during class lookup. (2) Extend descriptors with read-only/frozen-after-first-write behavior. (3) Support a documented storage API with a `threading.RLock` and concurrent stress tests. (4) Write a `unittest.TestCase` suite that parameterizes owner types. (5) Compare memory use for instance dictionaries versus external stores using `tracemalloc`, excluding notebook/debugger references. (6) Implement a multi-field transaction with rollback for arbitrary setter failures and specify its atomicity guarantees.

**Completion criterion:** every numbered check and both bonus labs print `PASS` when the notebook runs top-to-bottom in a fresh kernel. The outputs are assertions and explanations, not fixed object IDs or stale memory readings.